# Section 5: Cross-Domain Composition (Q41–Q50)

Multi-step queries combining BOM explosion, supplier lookup, transport routing, and financial data.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import (
    get_session, run_sql, explode_bom, bom_to_df,
    build_transport_graph, resolve_location_key, display_path
)
import networkx as nx
import pandas as pd
conn, ontology = get_session()
G = build_transport_graph(conn)

## Q41

For every ingredient in SKU-ORAL-001's BOM, find all suppliers who can provide it, their unit cost, and lead time. Flag any supplier records that look like duplicates — I've seen "-ALT" entries that are probably the same vendor.

In [ ]:
bom = explode_bom(conn, sku_code='SKU-ORAL-001')
ingredient_ids = list({r['ingredient_id'] for r in bom if not r.get('is_intermediate')})
placeholders = ','.join(['%s'] * len(ingredient_ids))

df = run_sql(conn, f"""
    SELECT i.ingredient_code, i.name as ingredient_name,
           s.supplier_code, s.name as supplier_name,
           si.unit_cost, si.lead_time_days, si.min_order_qty,
           CASE WHEN s.supplier_code LIKE '%-ALT' THEN 'POSSIBLE DUPLICATE' ELSE '' END as flag
    FROM supplier_ingredients si
    JOIN ingredients i ON si.ingredient_id = i.id
    JOIN suppliers s ON si.supplier_id = s.id
    WHERE si.ingredient_id IN ({placeholders})
    ORDER BY i.ingredient_code, si.unit_cost
""", ingredient_ids)
display(df)
dupes = df[df['flag'] == 'POSSIBLE DUPLICATE']
print(f"\nFlagged as possible duplicates: {len(dupes)}")

## Q42

What is the fully landed cost to produce one batch of SKU-ORAL-001 at the Dallas plant (PLANT-TX)? That means the BOM raw material cost plus the inbound freight cost to get each ingredient from its cheapest supplier to Dallas via the shortest route.

In [ ]:
bom = explode_bom(conn, sku_code='SKU-ORAL-001', resolve_costs=True)
plant_tx = resolve_location_key(conn, 'PLANT-TX')

# Get supplier locations for cheapest suppliers
supplier_codes = list({r['cheapest_supplier_code'] for r in bom if r.get('cheapest_supplier_code')})
placeholders = ','.join(['%s'] * len(supplier_codes))
sup_locs = run_sql(conn, f"""
    SELECT supplier_code, id FROM suppliers WHERE supplier_code IN ({placeholders})
""", supplier_codes)
sup_map = dict(zip(sup_locs['supplier_code'], sup_locs['id']))

material_cost = 0
freight_entries = []
for r in bom:
    if r.get('is_intermediate'):
        continue
    line_cost = r['cumulative_quantity_kg'] * (r.get('cheapest_unit_cost') or 0)
    material_cost += line_cost
    
    sup_code = r.get('cheapest_supplier_code')
    if sup_code and sup_code in sup_map:
        sup_key = f"supplier:{sup_map[sup_code]}"
        try:
            dist = nx.shortest_path_length(G, sup_key, plant_tx, weight='distance_km')
            freight_entries.append({'ingredient': r['ingredient_code'], 'supplier': sup_code, 'distance_km': dist})
        except nx.NetworkXNoPath:
            freight_entries.append({'ingredient': r['ingredient_code'], 'supplier': sup_code, 'distance_km': None})

print(f"Raw material cost: ${material_cost:.2f}")
freight_df = pd.DataFrame(freight_entries)
if len(freight_df) > 0:
    print(f"\nInbound freight distances (km):")
    display(freight_df)
    total_freight_km = freight_df['distance_km'].dropna().sum()
    print(f"Total inbound freight distance: {total_freight_km:.1f} km")
    # Estimate freight cost at ~$0.05/km (illustrative)
    freight_cost = total_freight_km * 0.05
    print(f"Estimated freight cost (@$0.05/km): ${freight_cost:.2f}")
    print(f"\nTotal landed cost: ${material_cost + freight_cost:.2f}")

## Q43

SKU-PERSONAL-229 has a discontinued alias (SKU-PERSONAL-229-OLD). Across the entire alias pair — the old code and the current code — what is the total inventory sitting in our network right now? Break it out by location. I suspect we have cases booked under the stale code.

In [ ]:
# Find both SKU IDs
sku_ids = run_sql(conn, """
    SELECT id, sku_code, is_active FROM skus
    WHERE sku_code IN ('SKU-PERSONAL-229', 'SKU-PERSONAL-229-OLD')
""")
display(sku_ids)

ids = list(sku_ids['id'])
placeholders = ','.join(['%s'] * len(ids))

# Get latest inventory for both
inv = run_sql(conn, f"""
    SELECT s.sku_code, inv.location_type, inv.location_id, inv.quantity_cases, inv.day
    FROM inventory inv
    JOIN skus s ON inv.sku_id = s.id
    WHERE inv.sku_id IN ({placeholders})
      AND inv.day = (SELECT MAX(day) FROM inventory)
    ORDER BY inv.location_type, inv.location_id
""", ids)
display(inv)
print(f"\nTotal cases across both codes: {inv['quantity_cases'].sum()}")

## Q44

Which retail locations within two route hops of the Chicago RDC (RDC-MW) have orders still in "pending" status? I want to prioritize nearby fulfillment.

In [ ]:
rdc_mw = resolve_location_key(conn, 'RDC-MW')

# Find retail locations within 2 hops
nearby_stores = set()
for node in G.successors(rdc_mw):  # 1 hop
    if node.startswith('store:'):
        nearby_stores.add(node)
    for node2 in G.successors(node):  # 2 hops
        if node2.startswith('store:'):
            nearby_stores.add(node2)

store_ids = [int(s.split(':')[1]) for s in nearby_stores]
print(f"Retail locations within 2 hops of RDC-MW: {len(store_ids)}")

if store_ids:
    placeholders = ','.join(['%s'] * len(store_ids))
    df = run_sql(conn, f"""
        SELECT o.order_number, o.status, rl.location_code, rl.name as store_name
        FROM orders o
        JOIN retail_locations rl ON o.retail_location_id = rl.id
        WHERE o.status = 'pending'
          AND rl.id IN ({placeholders})
        ORDER BY o.order_number
    """, store_ids)
    display(df)
    print(f"\nPending orders near RDC-MW: {len(df)}")

## Q45

Which active work orders use formulas that contain Coconut Oil RBD (BLK-OIL-003) as an ingredient? We need to assess our production exposure to a potential coconut oil disruption.

In [ ]:
run_sql(conn, """
    SELECT wo.wo_number, wo.status, wo.planned_quantity_kg,
           f.formula_code, p.plant_code
    FROM work_orders wo
    JOIN formulas f ON wo.formula_id = f.id
    JOIN formula_ingredients fi ON fi.formula_id = f.id
    JOIN ingredients i ON fi.ingredient_id = i.id
    JOIN plants p ON wo.plant_id = p.id
    WHERE i.ingredient_code = 'BLK-OIL-003'
      AND wo.status IN ('planned', 'in_progress')
    ORDER BY wo.status, wo.wo_number
""")

## Q46

What is the total AP invoice amount for all ingredients that go into SKU-ORAL-005's bill of materials? Trace the BOM, find each ingredient's supplier, and sum their AP invoices for the year.

In [ ]:
bom = explode_bom(conn, sku_code='SKU-ORAL-005')
ingredient_ids = list({r['ingredient_id'] for r in bom if not r.get('is_intermediate')})
placeholders = ','.join(['%s'] * len(ingredient_ids))

df = run_sql(conn, f"""
    SELECT i.ingredient_code, i.name as ingredient_name,
           s.supplier_code, s.name as supplier_name,
           SUM(api.total_amount) as total_ap_amount,
           COUNT(api.id) as invoice_count
    FROM supplier_ingredients si
    JOIN ingredients i ON si.ingredient_id = i.id
    JOIN suppliers s ON si.supplier_id = s.id
    JOIN ap_invoices api ON api.supplier_id = s.id
    WHERE si.ingredient_id IN ({placeholders})
    GROUP BY i.ingredient_code, i.name, s.supplier_code, s.name
    ORDER BY total_ap_amount DESC
""", ingredient_ids)
display(df)
print(f"\nTotal AP exposure: ${df['total_ap_amount'].sum():,.2f}")

## Q47

We have a rush order at retail location STORE-RET-001-0100. Which DC currently has inventory of SKU-HOME-376, and what is the shortest route from that DC to the retail location? Find the optimal fulfillment source.

In [ ]:
# Find DCs with inventory of SKU-HOME-376
dc_inv = run_sql(conn, """
    SELECT dc.dc_code, dc.type, inv.quantity_cases, inv.day
    FROM inventory inv
    JOIN skus s ON inv.sku_id = s.id
    JOIN distribution_centers dc ON inv.location_id = dc.id
    WHERE s.sku_code = 'SKU-HOME-376'
      AND inv.location_type IN ('rdc', 'customer_dc')
      AND inv.quantity_cases > 0
      AND inv.day = (SELECT MAX(day) FROM inventory)
    ORDER BY inv.quantity_cases DESC
""")
print("DCs with SKU-HOME-376 inventory:")
display(dc_inv)

store_key = resolve_location_key(conn, 'STORE-RET-001-0100')
if len(dc_inv) > 0 and store_key:
    best_dist = float('inf')
    best_dc = None
    best_path = None
    for _, row in dc_inv.iterrows():
        dc_key = resolve_location_key(conn, row['dc_code'])
        if dc_key:
            try:
                dist = nx.shortest_path_length(G, dc_key, store_key, weight='distance_km')
                path = nx.shortest_path(G, dc_key, store_key, weight='distance_km')
                if dist < best_dist:
                    best_dist = dist
                    best_dc = row['dc_code']
                    best_path = path
            except nx.NetworkXNoPath:
                pass
    if best_dc:
        print(f"\nOptimal fulfillment: {best_dc} ({best_dist:.1f} km)")
        display(display_path(G, best_path, 'distance_km'))

## Q48

Show me all work orders in "in_progress" status that use formulas containing Glycerin USP (ACT-HUMECTANT-001). I need to know our active production commitment to glycerin-dependent products.

In [ ]:
run_sql(conn, """
    SELECT wo.wo_number, wo.status, wo.planned_quantity_kg,
           f.formula_code, p.plant_code,
           fi.quantity_kg as glycerin_per_batch
    FROM work_orders wo
    JOIN formulas f ON wo.formula_id = f.id
    JOIN formula_ingredients fi ON fi.formula_id = f.id
    JOIN ingredients i ON fi.ingredient_id = i.id
    JOIN plants p ON wo.plant_id = p.id
    WHERE i.ingredient_code = 'ACT-HUMECTANT-001'
      AND wo.status = 'in_progress'
    ORDER BY wo.wo_number
""")

## Q49

Trace the full supply chain for order ORD-100-GRO-DC-001-28: show the order lines, the SKUs on each line, the formula behind each SKU, every ingredient in those formulas, and the suppliers who provide them. Give me the full six-hop picture.

In [ ]:
run_sql(conn, """
    SELECT o.order_number,
           ol.line_number,
           s.sku_code,
           f.formula_code,
           i.ingredient_code, i.name as ingredient_name,
           sup.supplier_code, sup.name as supplier_name,
           si.unit_cost
    FROM orders o
    JOIN order_lines ol ON ol.order_id = o.id
    JOIN skus s ON ol.sku_id = s.id
    JOIN formulas f ON f.product_id = s.id AND f.bom_level = 0
    JOIN formula_ingredients fi ON fi.formula_id = f.id
    JOIN ingredients i ON fi.ingredient_id = i.id
    JOIN supplier_ingredients si ON si.ingredient_id = i.id
    JOIN suppliers sup ON si.supplier_id = sup.id
    WHERE o.order_number = 'ORD-100-GRO-DC-001-28'
    ORDER BY ol.line_number, i.ingredient_code, si.unit_cost
""")

## Q50

For each finished SKU in the Oral Care category, compare the total AR invoice revenue against the total BOM raw material cost. Which SKUs have the healthiest margin, and which are underwater?

In [ ]:
# AR revenue per SKU
revenue = run_sql(conn, """
    SELECT s.sku_code, s.name as sku_name,
           SUM(arl.line_amount) as total_revenue
    FROM ar_invoice_lines arl
    JOIN skus s ON arl.sku_id = s.id
    WHERE s.category = 'Oral Care'
    GROUP BY s.sku_code, s.name
""")

# BOM cost per SKU (using cheapest supplier for each ingredient)
oral_skus = run_sql(conn, "SELECT sku_code FROM skus WHERE category = 'Oral Care' AND is_active = true")
costs = []
for sku_code in oral_skus['sku_code']:
    bom = explode_bom(conn, sku_code=sku_code, resolve_costs=True)
    if bom:
        total_cost = sum(r['cumulative_quantity_kg'] * (r.get('cheapest_unit_cost') or 0) for r in bom)
        costs.append({'sku_code': sku_code, 'bom_cost_per_batch': total_cost})

cost_df = pd.DataFrame(costs)
merged = revenue.merge(cost_df, on='sku_code', how='outer')
merged['margin'] = merged['total_revenue'] - merged['bom_cost_per_batch'].fillna(0)
merged = merged.sort_values('margin', ascending=False)

print("Oral Care SKU margin analysis (revenue vs BOM cost):")
display(merged)

underwater = merged[merged['margin'] < 0]
print(f"\nSKUs underwater (negative margin): {len(underwater)}")

In [ ]:
conn.close()
print("Session closed.")